# Resilience Patterns — Circuit Breakers, Retry, Memory, and Human-in-the-Loop

**Multigen SDK — Notebook 19**

---

Production AI systems face constant adversity:
- LLM APIs have rate limits, timeouts, and occasional outages
- External tools (search APIs, databases) are flaky
- LLM outputs are non-deterministic — sometimes the first call gives garbage
- Long-running workflows need to maintain context across multiple turns
- Some decisions are too important to automate — a human must review them

The Multigen SDK provides **four resilience primitives** that wrap any agent with production-grade reliability:

| Primitive | Problem It Solves | Key Parameters |
|---|---|---|
| **RetryAgent** | Transient failures (rate limits, timeouts) | `max_attempts`, `base_delay`, `backoff_factor` |
| **CircuitBreakerAgent** | Cascading failures from repeated errors | `failure_threshold`, `recovery_timeout` |
| **MemoryAgent** | Context loss in multi-turn conversations | `window=N` |
| **HumanInLoopAgent** | Decisions requiring human judgment | `review_fn`, `timeout` |

---

### The Resilience Stack

These primitives **compose** — you can layer them for defense-in-depth:

```
HumanInLoop(
  CircuitBreaker(
    Retry(
      MemoryAgent(
        your_llm_agent
      )
    )
  )
)
```

Each wrapper adds a layer of protection without modifying the underlying agent.

## Section 1: Why Resilience Matters in Production

### Real Failure Scenarios

**Scenario 1: Rate Limits**
```
Request 1 → OpenAI → ✓ (fast)
Request 2 → OpenAI → ✓ (fast)
Request 3 → OpenAI → ✗ 429 Too Many Requests
Request 4 → OpenAI → ✗ 429 Too Many Requests
```
Without retry: your pipeline fails at step 3. With retry: it waits and succeeds.

**Scenario 2: Cascading Failure**
```
LLM API is down. Every request fails immediately.
Without circuit breaker: 1000 requests per second, all failing, all blocking
With circuit breaker: after 5 failures, STOP trying for 30 seconds → save resources
```

**Scenario 3: Multi-Turn Context Loss**
```
Turn 1: "My name is Alice, I'm building a trading bot."
Turn 2: "What should I name my variables?"  
→ Without memory: agent has no idea who you are or what you're building.
→ With MemoryAgent(window=5): agent remembers the last 5 turns.
```

**Scenario 4: High-Stakes Decision**
```
Agent analyzes contract and suggests: "Accept the $500K deal"
→ Without human review: auto-accepted. Risk of mistakes.
→ With HumanInLoopAgent: pauses, sends for review, awaits approval.
```

### Production System Requirements

For any AI system serving real users:
1. **Availability**: Don't fail permanently on transient errors → Retry
2. **Stability**: Don't hammer failing services → Circuit Breaker
3. **Context**: Maintain conversation state → Memory
4. **Safety**: Human oversight for consequential decisions → HumanInLoop

In [ ]:
# SDK Imports — all resilience primitives
import asyncio
import time
import random

from multigen import (
    CircuitBreakerAgent,
    RetryAgent,
    MemoryAgent,
    HumanInLoopAgent,
    FunctionAgent,
    Chain
)

print("All resilience primitives imported successfully:")
print("  RetryAgent, CircuitBreakerAgent, MemoryAgent, HumanInLoopAgent")

## Section 2: RetryAgent — Exponential Backoff

`RetryAgent` wraps any agent with **automatic retry on failure** using configurable exponential backoff.

### How It Works

When the wrapped agent raises an exception:
1. Wait `base_delay` seconds
2. Retry
3. If still fails: wait `base_delay × backoff_factor` seconds
4. Retry again
5. Continue until `max_attempts` is exhausted
6. If all attempts fail: re-raise the last exception

### Constructor Parameters

| Parameter | Default | Description |
|---|---|---|
| `agent` | required | The agent to wrap |
| `max_attempts` | 3 | Total attempts (1 original + N-1 retries) |
| `base_delay` | 1.0 | Seconds to wait before first retry |
| `backoff_factor` | 2.0 | Multiplier for each subsequent delay |
| `jitter` | True | Add randomness to avoid thundering herd |
| `retry_on` | [Exception] | Which exception types trigger retry |

In [ ]:
# --- RetryAgent construction and parameter explanation ---

# A simple reliable agent — won't fail
async def reliable_llm_call(prompt, context=None):
    """Simulates a reliable LLM API call."""
    await asyncio.sleep(0.01)
    return f"LLM response to: {prompt}"

reliable_agent = FunctionAgent(reliable_llm_call, name="ReliableLLM")

# Wrap with RetryAgent
retry_agent = RetryAgent(
    agent=reliable_agent,
    max_attempts=5,        # Try up to 5 times total
    base_delay=0.5,        # Wait 0.5s before first retry
    backoff_factor=2.0,    # Each retry waits 2× longer (0.5, 1.0, 2.0, 4.0 seconds)
    jitter=True,           # Add ±20% random jitter to avoid thundering herd
    retry_on=[Exception]   # Retry on any exception
)

print("RetryAgent configured:")
print(f"  Wrapped agent: {retry_agent.agent.name}")
print(f"  max_attempts:  {retry_agent.max_attempts}")
print(f"  base_delay:    {retry_agent.base_delay}s")
print(f"  backoff_factor:{retry_agent.backoff_factor}")
print(f"  jitter:        {retry_agent.jitter}")
print()
print("Delay schedule (without jitter):")
delay = retry_agent.base_delay
for attempt in range(1, retry_agent.max_attempts):
    print(f"  After attempt {attempt} failure: wait {delay:.2f}s")
    delay *= retry_agent.backoff_factor
print(f"  After attempt {retry_agent.max_attempts} failure: GIVE UP (re-raise exception)")

## Section 3: RetryAgent Demo — Wrapping a Flaky Agent

Let's create a "flaky" agent that fails the first 3 times, then succeeds on attempt 4. The RetryAgent should handle this transparently.

In [ ]:
# --- RetryAgent demo with a flaky agent ---

# A stateful flaky agent that fails N times before succeeding
class FlakyAgent:
    """
    Simulates an LLM API that:
    - Fails with RateLimitError the first `fail_count` times
    - Succeeds on attempt `fail_count + 1`
    
    This models a real API that's temporarily rate-limited.
    """
    
    def __init__(self, fail_count=3):
        self.fail_count = fail_count
        self.call_count = 0
        self.name = "FlakyLLMAgent"
    
    async def run(self, prompt, context=None):
        self.call_count += 1
        
        if self.call_count <= self.fail_count:
            error_msg = f"429 Too Many Requests (attempt {self.call_count}/{self.fail_count})"
            print(f"    [FlakyAgent] Attempt {self.call_count}: FAILING → {error_msg}")
            raise ConnectionError(error_msg)
        
        print(f"    [FlakyAgent] Attempt {self.call_count}: SUCCESS")
        return {
            "response": f"LLM response after {self.call_count} attempts: {prompt}",
            "attempts_needed": self.call_count,
            "model": "gpt-4"
        }

# Create the flaky agent (fails first 3 times)
flaky = FlakyAgent(fail_count=3)

# Wrap it with RetryAgent — set delays to 0.1s for demo speed
retry_wrapped = RetryAgent(
    agent=flaky,
    max_attempts=5,       # Up to 5 attempts — enough to cover 3 failures + 1 success
    base_delay=0.1,       # Short delays for demo (production would use 1.0+)
    backoff_factor=2.0,
    jitter=False          # Disable jitter for predictable demo output
)

async def demo_retry_flaky():
    print("=" * 60)
    print("RETRY DEMO — Flaky agent fails 3 times, succeeds on attempt 4")
    print("=" * 60)
    print()
    print("Configuration:")
    print(f"  Agent fails first 3 calls with ConnectionError")
    print(f"  RetryAgent: max_attempts=5, base_delay=0.1s, backoff=2.0×")
    print()
    print("Expected retry schedule:")
    print("  Attempt 1 → FAIL → wait 0.1s")
    print("  Attempt 2 → FAIL → wait 0.2s")
    print("  Attempt 3 → FAIL → wait 0.4s")
    print("  Attempt 4 → SUCCESS")
    print()
    
    start = time.time()
    
    # This call will transparently retry 3 times before succeeding
    result = await retry_wrapped.run("Explain backpressure in distributed systems")
    
    elapsed = time.time() - start
    
    print()
    print("=" * 60)
    print("RESULT")
    print("=" * 60)
    print(f"result: {result}")
    print(f"Total wall time: {elapsed:.2f}s (0.1 + 0.2 + 0.4 = 0.7s in delays)")
    print()
    print("From the caller's perspective: one call, one result.")
    print("The retry logic is completely transparent.")

asyncio.run(demo_retry_flaky())

In [ ]:
# --- What happens when max_attempts is exhausted? ---

# Agent fails EVERY time — will exhaust all retries
always_fails = FlakyAgent(fail_count=100)  # Fails 100 times — more than max_attempts

retry_exhausted = RetryAgent(
    agent=always_fails,
    max_attempts=3,      # Only 3 attempts
    base_delay=0.05,
    backoff_factor=2.0,
    jitter=False
)

async def demo_retry_exhausted():
    print("=" * 60)
    print("RETRY EXHAUSTED — all attempts fail")
    print("=" * 60)
    print()
    print("Agent fails EVERY time. RetryAgent has max_attempts=3.")
    print("After 3 failures, the exception is re-raised to the caller.")
    print()
    
    try:
        result = await retry_exhausted.run("This will fail")
    except ConnectionError as e:
        print(f"Exception caught after all retries: {type(e).__name__}: {e}")
        print()
        print("The caller must handle this exception — wrap in try/except")
        print("or let it propagate to the Circuit Breaker (see next section).")
    
    print()
    print(f"Total calls made to underlying agent: {always_fails.call_count}")
    print(f"(= max_attempts = 3)")

asyncio.run(demo_retry_exhausted())

## Section 4: Circuit Breaker Concept

The **Circuit Breaker** pattern is named after electrical circuit breakers. Just as an electrical breaker trips to prevent damage when current is too high, a software circuit breaker "trips" to prevent cascading failures.

### Three States

```
                   failures >= threshold
  ┌─────────┐      ──────────────────►      ┌──────────┐
  │  CLOSED  │  ← normal operation →        │   OPEN   │  ← all calls rejected fast
  │(passing) │                              │(tripped) │
  └─────────┘                              └──────────┘
       ▲                                          │
       │  success                                 │ after recovery_timeout
       │           ┌────────────┐                 ▼
       └───────────│ HALF_OPEN  │◄───────────────┘
                   │(testing)   │  allow one test call
                   └────────────┘
                         │ failure
                         ▼
                      back to OPEN
```

### State Transitions

| From | To | When |
|---|---|---|
| **CLOSED** | **OPEN** | `failures_in_window >= failure_threshold` |
| **OPEN** | **HALF_OPEN** | `time_since_open >= recovery_timeout` |
| **HALF_OPEN** | **CLOSED** | Test call succeeds |
| **HALF_OPEN** | **OPEN** | Test call fails (still broken) |

### Why This Helps

Without circuit breaker:
```
API is down → 10,000 requests/min all timeout (30s each) → 300,000 thread-seconds wasted
```

With circuit breaker:
```
API is down → 5 failures → circuit OPENS → requests rejected in <1ms → resources saved
```

## Section 5: CircuitBreakerAgent

`CircuitBreakerAgent` wraps any agent with the circuit breaker pattern.

### Constructor Parameters

| Parameter | Default | Description |
|---|---|---|
| `agent` | required | The agent to protect |
| `failure_threshold` | 5 | Number of failures to trip the circuit |
| `recovery_timeout` | 30.0 | Seconds to wait before trying HALF_OPEN |
| `success_threshold` | 1 | Successes in HALF_OPEN needed to re-close |
| `window_size` | 60.0 | Time window for counting failures (seconds) |

In [ ]:
# --- CircuitBreakerAgent construction ---

async def llm_api_fn(prompt, context=None):
    """Simulates an LLM API call that might fail."""
    await asyncio.sleep(0.01)
    return f"LLM response to: {prompt}"

llm_agent = FunctionAgent(llm_api_fn, name="LLMAgent")

# Create a circuit breaker around the LLM agent
cb_agent = CircuitBreakerAgent(
    agent=llm_agent,
    failure_threshold=3,    # Open circuit after 3 failures
    recovery_timeout=5.0,   # Wait 5 seconds before trying again (short for demo)
    success_threshold=1,    # One success in HALF_OPEN → back to CLOSED
    window_size=60.0        # Count failures within a 60-second window
)

print("CircuitBreakerAgent configured:")
print(f"  Wrapped agent:      {cb_agent.agent.name}")
print(f"  failure_threshold:  {cb_agent.failure_threshold}")
print(f"  recovery_timeout:   {cb_agent.recovery_timeout}s")
print(f"  success_threshold:  {cb_agent.success_threshold}")
print(f"  window_size:        {cb_agent.window_size}s")
print()

# Inspect the initial circuit state
print(f"Initial circuit_state: {cb_agent.circuit_state}")
# Should be 'CLOSED' — circuit is healthy, requests pass through

## Section 6: Circuit Breaker Demo — Triggering Failures

Let's trigger failures to watch the circuit trip from CLOSED to OPEN, then observe the HALF_OPEN recovery.

In [ ]:
# --- Circuit breaker state machine demo ---

# Counter to control when the "API" fails
api_call_counter = {"count": 0}

async def unstable_api_fn(prompt, context=None):
    """
    Simulates an unstable API:
    - Calls 1-3: FAIL (triggers circuit OPEN)
    - Calls 4-5: BLOCKED by circuit (should not reach here)
    - After recovery_timeout: SUCCEED (allows HALF_OPEN → CLOSED)
    """
    api_call_counter["count"] += 1
    count = api_call_counter["count"]
    
    if count <= 3:
        raise RuntimeError(f"API Error: Service unavailable (call #{count})")
    
    return {"response": f"API recovered! Call #{count}", "status": "ok"}

unstable_agent = FunctionAgent(unstable_api_fn, name="UnstableAPI")

# Create circuit breaker with short recovery for demo
cb_demo = CircuitBreakerAgent(
    agent=unstable_agent,
    failure_threshold=3,     # Open after 3 failures
    recovery_timeout=1.0,    # Only 1 second recovery time for demo
    success_threshold=1
)

async def demo_circuit_breaker():
    print("=" * 60)
    print("CIRCUIT BREAKER STATE TRANSITIONS")
    print("=" * 60)
    print()
    
    # Phase 1: Force 3 failures to trip the circuit
    print("Phase 1: Sending 3 requests that all fail")
    print(f"Initial state: {cb_demo.circuit_state}")
    print()
    
    for i in range(3):
        try:
            result = await cb_demo.run(f"Request {i+1}")
            print(f"  Request {i+1}: SUCCESS → {result}")
        except RuntimeError as e:
            print(f"  Request {i+1}: FAILED → {e}")
            print(f"             Circuit state: {cb_demo.circuit_state}")
    
    print()
    print(f"Circuit state after 3 failures: {cb_demo.circuit_state}")
    assert cb_demo.circuit_state == "OPEN", "Circuit should be OPEN now!"
    print("✓ Circuit is OPEN — no more calls will reach the underlying API")
    
    # Phase 2: Try calling with OPEN circuit — should fail fast (no wait)
    print()
    print("Phase 2: Calling with OPEN circuit (should fail instantly)")
    for i in range(2):
        try:
            start = time.time()
            result = await cb_demo.run(f"Blocked request {i+1}")
        except Exception as e:
            elapsed = time.time() - start
            print(f"  Blocked request {i+1}: {type(e).__name__}: {e}")
            print(f"             Failed in {elapsed*1000:.1f}ms (fast-fail, no API call)")
    
    print()
    print(f"API was called {api_call_counter['count']} times total")
    print("(Only 3 times — the 2 blocked requests never reached the API)")
    
    # Phase 3: Wait for recovery_timeout, then circuit moves to HALF_OPEN
    print()
    print("Phase 3: Waiting 1.1 seconds for recovery_timeout to expire...")
    await asyncio.sleep(1.1)
    
    # The first call after recovery_timeout will be a test call (HALF_OPEN)
    print(f"Circuit state after wait: {cb_demo.circuit_state}")
    # Should be HALF_OPEN (SDK checks when a call arrives)
    
    # Phase 4: Make a test call — this one should succeed (our API recovers after 3 failures)
    print()
    print("Phase 4: Test call in HALF_OPEN state")
    try:
        result = await cb_demo.run("Test call after recovery")
        print(f"  Test call: SUCCESS → {result}")
        print(f"  Circuit state: {cb_demo.circuit_state}")
        # Should be CLOSED again
    except Exception as e:
        print(f"  Test call: FAILED → {e}  (circuit would re-OPEN)")
    
    print()
    print("Summary:")
    print("  CLOSED  → received 3 failures → tripped to OPEN")
    print("  OPEN    → 2 fast-fail rejections → waited recovery_timeout")
    print("  HALF_OPEN → 1 test call succeeded → returned to CLOSED")

asyncio.run(demo_circuit_breaker())

## Section 7: cb.circuit_state — Inspecting Current State

You can inspect and monitor the circuit breaker state at any time via `cb_agent.circuit_state`.

This is useful for:
- Health check endpoints (is the LLM API healthy?)
- Dashboards and monitoring
- Conditional logic (if circuit is open, use a fallback)

In [ ]:
# --- Inspecting circuit state and acting on it ---

monitoring_cb = CircuitBreakerAgent(
    agent=FunctionAgent(lambda x, ctx=None: asyncio.sleep(0), name="DummyAgent"),
    failure_threshold=3,
    recovery_timeout=30.0
)

# The circuit_state property returns a string: 'CLOSED', 'OPEN', or 'HALF_OPEN'
print("Circuit state inspection:")
print(f"  current state: {monitoring_cb.circuit_state}")
print()

# Use in a health check
def health_check(cb: CircuitBreakerAgent) -> dict:
    """Returns health status of the wrapped service."""
    state = cb.circuit_state
    return {
        "service": cb.agent.name,
        "circuit_state": state,
        "healthy": state == "CLOSED",
        "accepting_requests": state in ("CLOSED", "HALF_OPEN"),
        "status": {
            "CLOSED":    "Healthy — all requests passing through",
            "OPEN":      "Unhealthy — rejecting all requests fast-fail",
            "HALF_OPEN": "Recovering — test traffic only"
        }.get(state, "Unknown")
    }

health = health_check(monitoring_cb)
print("Health check result:")
for k, v in health.items():
    print(f"  {k}: {v}")

print()

# Conditional logic based on circuit state
async def smart_call(cb: CircuitBreakerAgent, prompt: str, fallback_fn):
    """
    Smart wrapper: if circuit is OPEN, use fallback immediately.
    Otherwise try the primary agent.
    """
    if cb.circuit_state == "OPEN":
        print(f"  Circuit is OPEN — using fallback immediately for: {prompt!r}")
        return await fallback_fn(prompt)
    
    try:
        return await cb.run(prompt)
    except Exception as e:
        print(f"  Call failed: {e}. Using fallback.")
        return await fallback_fn(prompt)

async def fallback_fn(prompt):
    """Simple fallback: return a cached/default response."""
    return {"response": f"FALLBACK: {prompt}", "source": "cache"}

async def demo_state_inspection():
    result = await smart_call(monitoring_cb, "What is the capital of France?", fallback_fn)
    print(f"Result: {result}")

asyncio.run(demo_state_inspection())

## Section 8: Wrapping CircuitBreaker Around Retry — Layered Resilience

The most resilient configuration combines both:

```
CircuitBreakerAgent(
    RetryAgent(
        your_agent
    )
)
```

**How they interact:**
1. A call comes in → CircuitBreaker checks if circuit is CLOSED
2. If CLOSED → passes to RetryAgent
3. RetryAgent attempts the call up to `max_attempts` times
4. If all retries fail → raises exception
5. CircuitBreaker counts this as ONE failure (regardless of how many retries)
6. After `failure_threshold` failures → circuit opens

This way:
- Transient errors (1-3 failures): RetryAgent handles silently
- Persistent failures: Circuit breaker opens to prevent resource waste

In [ ]:
# --- Layered: CircuitBreaker wrapping Retry ---

# Create a reliable base agent
async def base_agent_fn(prompt, context=None):
    await asyncio.sleep(0.01)
    return f"Response to: {prompt}"

base_agent = FunctionAgent(base_agent_fn, name="BaseAgent")

# Inner layer: retry on transient failures
retry_layer = RetryAgent(
    agent=base_agent,
    max_attempts=3,
    base_delay=0.05,
    backoff_factor=2.0,
    jitter=False
)

# Outer layer: circuit breaker for persistent failures
cb_retry_stack = CircuitBreakerAgent(
    agent=retry_layer,         # Wraps the RetryAgent
    failure_threshold=2,        # Open after 2 "all-retries-exhausted" failures
    recovery_timeout=2.0
)

print("Layered resilience stack:")
print("  CircuitBreakerAgent")
print("    └── RetryAgent (max_attempts=3, base_delay=0.05s)")
print("          └── BaseAgent")
print()
print("Behavior:")
print("  Failure scenario A (transient):")
print("    Call → CB passes → Retry attempt 1 fails → attempt 2 succeeds → CB counts 0 failures")
print()
print("  Failure scenario B (persistent):")
print("    Call → CB passes → Retry attempt 1,2,3 ALL fail → CB counts 1 failure")
print("    After 2 such failures → CB opens")
print()
print("  When CB is OPEN:")
print("    Call → CB rejects immediately (never reaches RetryAgent or BaseAgent)")
print("    Resources saved: no waiting, no retrying a broken service")

async def demo_layered():
    # Normal operation
    result = await cb_retry_stack.run("Test layered resilience")
    print(f"\nNormal call result: {result}")
    print(f"Circuit state: {cb_retry_stack.circuit_state}")

asyncio.run(demo_layered())

## Section 9: MemoryAgent — Windowed Conversation History

`MemoryAgent` adds **conversation memory** to any agent.

Without memory, every call to an LLM agent is stateless — the agent has no knowledge of previous turns. With MemoryAgent, each call includes the last N turns of conversation history in the input.

### How It Works

1. MemoryAgent maintains a **sliding window** of the last `window` turns
2. On each call: `history + new_input` is passed to the wrapped agent
3. The wrapped agent's response is appended to the history
4. When history exceeds `window` turns: oldest turns are dropped

### Constructor Parameters

| Parameter | Default | Description |
|---|---|---|
| `agent` | required | The agent to add memory to |
| `window` | 10 | Maximum number of turns to remember |
| `include_system_prompt` | False | Prepend a system prompt to every call |

In [ ]:
# --- MemoryAgent construction ---

async def chatbot_fn(conversation_input, context=None):
    """
    Simulates a conversational LLM agent.
    The 'conversation_input' includes the history window + the new message.
    In production, this would be sent to OpenAI/Anthropic as the messages array.
    """
    if isinstance(conversation_input, dict):
        history = conversation_input.get("history", [])
        new_message = conversation_input.get("message", "")
    else:
        history = []
        new_message = str(conversation_input)
    
    # Simulate context-aware responses
    context_summary = f" (with {len(history)} turns of context)" if history else " (no context)"
    
    # A very simple "echo" chatbot that references history
    if history and "name" in str(history).lower():
        names = [h.get("user", "") for h in history if "name" in h.get("user", "").lower()]
        response = f"I remember your name from our earlier conversation. Responding to: {new_message}"
    elif "name" in new_message.lower():
        response = f"Got it, I'll remember your name! Responding to: {new_message}"
    else:
        response = f"Chatbot response to '{new_message}'{context_summary}"
    
    return response

chatbot_agent = FunctionAgent(chatbot_fn, name="ChatbotAgent")

# Wrap with MemoryAgent — remember last 5 turns
memory_chatbot = MemoryAgent(
    agent=chatbot_agent,
    window=5  # Remember last 5 turns of conversation
)

print("MemoryAgent configured:")
print(f"  Wrapped agent: {memory_chatbot.agent.name}")
print(f"  window:        {memory_chatbot.window} turns")
print()
print("Each call will include up to 5 previous (user, assistant) turn pairs.")
print("Turn 6 will drop turn 1 from the context window.")

## Section 10: Multi-Turn Conversation Demo

Let's have a 5-turn conversation and watch how history accumulates.

In [ ]:
# --- 5-turn conversation demo ---

async def demo_multiturn():
    print("=" * 60)
    print("MULTI-TURN CONVERSATION — 5 turns with MemoryAgent(window=5)")
    print("=" * 60)
    print()
    
    turns = [
        "Hello! My name is Alice and I'm a software engineer.",
        "I'm building a Python application for stock analysis.",
        "What naming conventions should I use for my functions?",
        "Should I use type hints throughout?",
        "What about async vs sync design? My app fetches live data."
    ]
    
    for i, turn in enumerate(turns, 1):
        print(f"Turn {i}:")
        print(f"  User: {turn}")
        
        # Call the memory-enabled chatbot
        result = await memory_chatbot.run(turn)
        
        print(f"  Bot:  {result}")
        
        # Inspect the memory state after this turn
        history_len = result["history_length"] if isinstance(result, dict) else i
        print(f"  [Memory: {history_len} turns stored]")
        print()
    
    # Show the stored history
    print("=" * 60)
    print("STORED HISTORY (from result metadata)")
    print("=" * 60)
    
    # Run one more turn to get the full history in the result
    final_result = await memory_chatbot.run("Summarize what we discussed.")
    
    if isinstance(final_result, dict):
        history = final_result.get("history", [])
        print(f"\nHistory window ({len(history)} turns):")
        for j, entry in enumerate(history, 1):
            user = entry.get("user", "?")[:60] if isinstance(entry, dict) else str(entry)[:60]
            bot = entry.get("assistant", "?")[:60] if isinstance(entry, dict) else ""
            print(f"  Turn {j}:")
            print(f"    User: {user}")
            print(f"    Bot:  {bot}")

asyncio.run(demo_multiturn())

## Section 11: Accessing Conversation History

When `MemoryAgent` returns results, it enriches the output with history metadata:
- **`result["history"]`** — the list of stored turns
- **`result["history_length"]`** — number of turns currently in the window

You can also access the history directly via `memory_agent.history`.

In [ ]:
# --- Accessing conversation history ---

async def demo_history_access():
    print("=" * 60)
    print("ACCESSING CONVERSATION HISTORY")
    print("=" * 60)
    print()
    
    # Create a fresh MemoryAgent
    fresh_memory = MemoryAgent(agent=chatbot_agent, window=3)
    
    # Have a 3-turn conversation
    messages = ["Turn 1 message", "Turn 2 message", "Turn 3 message"]
    
    for msg in messages:
        result = await fresh_memory.run(msg)
    
    # Access history from the result object
    if isinstance(result, dict):
        print("From result object:")
        print(f"  result['history']:        {result.get('history', 'N/A')}")
        print(f"  result['history_length']: {result.get('history_length', 'N/A')}")
    
    print()
    
    # Access history directly from the MemoryAgent
    print("From memory_agent.history:")
    history = fresh_memory.history
    print(f"  Type: {type(history)}")
    print(f"  Length: {len(history)} entries")
    
    for i, entry in enumerate(history):
        print(f"  Entry {i}: {entry}")

asyncio.run(demo_history_access())

## Section 12: memory_agent.clear_memory()

Call `memory_agent.clear_memory()` to **reset the conversation history**.

Use cases:
- A user ends a session — clear memory so the next user starts fresh
- A workflow completes — reset for the next workflow run
- Testing — ensure each test starts with a clean state

In [ ]:
# --- clear_memory() demo ---

async def demo_clear_memory():
    print("=" * 60)
    print("CLEAR_MEMORY() DEMONSTRATION")
    print("=" * 60)
    
    mem_agent = MemoryAgent(agent=chatbot_agent, window=5)
    
    # Session 1: Alice's conversation
    print("\n--- Session 1: Alice ---")
    await mem_agent.run("Hi, I'm Alice")
    await mem_agent.run("I work in finance")
    await mem_agent.run("What's a good Excel formula for CAGR?")
    
    print(f"History length after session 1: {len(mem_agent.history)} turns")
    print(f"History preview: {[str(h)[:40] for h in mem_agent.history]}")
    
    # Alice's session ends — clear memory
    print("\nAlice's session ended. Clearing memory...")
    mem_agent.clear_memory()
    
    print(f"History length after clear: {len(mem_agent.history)} turns")
    assert len(mem_agent.history) == 0, "Memory should be empty!"
    print("✓ Memory cleared successfully")
    
    # Session 2: Bob's conversation — starts completely fresh
    print("\n--- Session 2: Bob (fresh start) ---")
    result = await mem_agent.run("Hi, I'm Bob")
    print(f"Bob's first turn result: {result}")
    print(f"History length: {len(mem_agent.history)} (only Bob's turn, not Alice's)")
    
    # Verify there's no trace of Alice's context
    history_str = str(mem_agent.history)
    print(f"\n'Alice' in history? {'Alice' in history_str}")
    print("✓ Alice's context is gone — Bob starts fresh")

asyncio.run(demo_clear_memory())

## Section 13: HumanInLoopAgent — Pause for Human Review

`HumanInLoopAgent` pauses workflow execution and waits for a **human to review and approve** the agent's output before continuing.

This is essential for:
- **High-stakes decisions**: Publishing content, executing financial transactions, sending emails to customers
- **Compliance**: Regulatory requirements for human oversight in AI-generated content
- **Quality gates**: Ensuring AI outputs meet standards before being used

### How It Works

1. The wrapped agent runs and produces output
2. `HumanInLoopAgent` calls `review_fn(output)` — your review function
3. The `review_fn` can:
   - Return `True` → approved, continue execution
   - Return `False` → rejected, raise exception
   - Return a modified output → approved with edits
4. Optional `timeout` — if reviewer doesn't respond in time, fail or auto-approve

### Mock Review Function

In tests and demos, use a mock review function. In production, `review_fn` would:
- Send a Slack message with approve/reject buttons
- Show a web UI form
- Send an email with a clickable approval link
- Write to a database and poll for the review decision

In [ ]:
# --- HumanInLoopAgent with mock review functions ---

# The agent that produces content for review
async def content_generator_fn(topic, context=None):
    """Generates a marketing email draft for human review."""
    await asyncio.sleep(0.01)
    return {
        "subject": f"Special offer: {topic}",
        "body": f"Dear customer, we have an exclusive offer about {topic}. Act now!",
        "recipients": 50000,
        "urgency": "high"
    }

content_generator = FunctionAgent(content_generator_fn, name="EmailGenerator")

# --- Mock review function 1: Auto-approve ---
async def auto_approve_fn(output):
    """
    Mock reviewer that automatically approves all outputs.
    Use this in testing when you want the pipeline to run without interruption.
    """
    print(f"  [AutoReviewer] Reviewing output: {str(output)[:80]}")
    print(f"  [AutoReviewer] APPROVED (auto-approve mode)")
    return True  # Approved

# --- Mock review function 2: Quality-based approval ---
async def quality_review_fn(output):
    """
    Mock reviewer that checks content quality criteria.
    Rejects if the email body is too short or contains suspicious keywords.
    """
    if isinstance(output, dict):
        body = output.get("body", "")
        recipients = output.get("recipients", 0)
        
        print(f"  [QualityReviewer] Checking: {recipients} recipients, body length: {len(body)}")
        
        # Reject if too many recipients without explicit approval
        if recipients > 100000:
            print(f"  [QualityReviewer] REJECTED: {recipients} recipients exceeds limit")
            return False  # Rejected
        
        # Reject if body contains spam-like phrases
        spam_words = ["act now", "limited time", "click here"]
        body_lower = body.lower()
        for word in spam_words:
            if word in body_lower:
                print(f"  [QualityReviewer] REJECTED: Contains spam phrase '{word}'")
                return False
        
        print(f"  [QualityReviewer] APPROVED: Content looks clean")
        return True
    
    print(f"  [QualityReviewer] APPROVED: (non-dict output, passing through)")
    return True

# --- Mock review function 3: Edit and approve ---
async def edit_and_approve_fn(output):
    """
    Reviewer that improves the content before approving.
    Returns a modified version instead of True/False.
    """
    if isinstance(output, dict):
        # Edit: remove spam-like phrases
        edited = output.copy()
        edited["body"] = output["body"].replace("Act now!", "Learn more.")
        edited["reviewed"] = True
        edited["reviewer"] = "HumanEditor"
        print(f"  [EditReviewer] Edited and APPROVED")
        return edited  # Return the edited version
    return True

print("HumanInLoopAgent review functions defined:")
print("  auto_approve_fn   — always approves (for testing)")
print("  quality_review_fn — rejects spam, approves clean content")
print("  edit_and_approve_fn — edits content before approving")

In [ ]:
# --- HumanInLoopAgent demo ---

async def demo_human_in_loop():
    print("=" * 60)
    print("HUMAN-IN-THE-LOOP DEMO")
    print("=" * 60)
    
    # --- Scenario 1: Auto-approve ---
    print("\nScenario 1: Auto-approve (testing mode)")
    hil_auto = HumanInLoopAgent(
        agent=content_generator,
        review_fn=auto_approve_fn,
        timeout=30.0  # Wait up to 30 seconds for review
    )
    
    result = await hil_auto.run("Summer sale 2025")
    print(f"  Result: {result}")
    
    # --- Scenario 2: Quality-based (will fail) ---
    print("\nScenario 2: Quality review — content has 'act now' (spam)")
    hil_quality = HumanInLoopAgent(
        agent=content_generator,
        review_fn=quality_review_fn,
        timeout=30.0
    )
    
    try:
        result = await hil_quality.run("Summer sale 2025")  # body has 'Act now!'
        print(f"  Result: {result}")
    except PermissionError as e:
        print(f"  REJECTED by reviewer: {e}")
        print(f"  Email was NOT sent — human review blocked it")
    
    # --- Scenario 3: Edit and approve ---
    print("\nScenario 3: Edit and approve — reviewer improves content")
    hil_edit = HumanInLoopAgent(
        agent=content_generator,
        review_fn=edit_and_approve_fn,
        timeout=30.0
    )
    
    result = await hil_edit.run("Winter clearance event")
    print(f"  Result (after human edits): {result}")
    if isinstance(result, dict):
        print(f"  Body: {result.get('body')}")
        print(f"  Reviewed: {result.get('reviewed')}")
        print(f"  Reviewer: {result.get('reviewer')}")

asyncio.run(demo_human_in_loop())

## Section 14: Combining All Patterns

The most robust production configuration layers **all four resilience primitives**:

```python
HumanInLoopAgent(
    CircuitBreakerAgent(
        RetryAgent(
            MemoryAgent(
                your_llm_agent,
                window=5
            ),
            max_attempts=3
        ),
        failure_threshold=5
    ),
    review_fn=quality_gate_fn
)
```

And we can put this wrapped agent inside a `Chain` for multi-stage workflows.

In [ ]:
# --- Combined resilience stack ---

# Base LLM agent
async def production_llm_fn(messages, context=None):
    """Simulates a production LLM call with conversation history."""
    await asyncio.sleep(0.02)
    if isinstance(messages, dict):
        history = messages.get("history", [])
        current = messages.get("message", "")
    else:
        history = []
        current = str(messages)
    
    return {
        "response": f"[LLM] Answering: {current} (context: {len(history)} turns)",
        "model": "gpt-4",
        "tokens_used": len(str(current).split()) * 4
    }

base_llm = FunctionAgent(production_llm_fn, name="ProductionLLM")

# Layer 1: Memory (innermost — attached to the LLM)
memory_llm = MemoryAgent(
    agent=base_llm,
    window=5  # Remember last 5 turns
)

# Layer 2: Retry (handles transient LLM API failures)
retry_memory_llm = RetryAgent(
    agent=memory_llm,
    max_attempts=3,
    base_delay=0.1,
    backoff_factor=2.0
)

# Layer 3: Circuit Breaker (prevents cascade on persistent LLM outage)
cb_retry_memory_llm = CircuitBreakerAgent(
    agent=retry_memory_llm,
    failure_threshold=5,
    recovery_timeout=30.0
)

# Layer 4: Human-in-Loop review function for high-stakes outputs
async def stakes_review_fn(output):
    """Review function: approves most responses, flags anything too long."""
    if isinstance(output, dict):
        response = output.get("response", "")
        tokens = output.get("tokens_used", 0)
        
        if tokens > 1000:  # Flag very long responses
            print(f"  [HumanReview] FLAGGED: Response too long ({tokens} tokens) — review needed")
            # In production: send to Slack for review
            # Here: auto-approve after flagging
            return True
    
    return True  # Approve

# Full resilience stack
full_stack = HumanInLoopAgent(
    agent=cb_retry_memory_llm,
    review_fn=stakes_review_fn,
    timeout=60.0
)

print("Full resilience stack assembled:")
print("  HumanInLoopAgent")
print("    └── CircuitBreakerAgent (failure_threshold=5, recovery=30s)")
print("          └── RetryAgent (max_attempts=3, base_delay=0.1s)")
print("                └── MemoryAgent (window=5)")
print("                      └── ProductionLLM")
print()

# Put the full stack in a Chain with a pre-processing agent
async def preprocessor_fn(raw_input, context=None):
    """Preprocesses user input before sending to LLM."""
    await asyncio.sleep(0.005)
    if isinstance(raw_input, str):
        return {"message": raw_input.strip(), "source": "user", "preprocessed": True}
    return raw_input

async def postprocessor_fn(llm_output, context=None):
    """Post-processes LLM output before returning to user."""
    await asyncio.sleep(0.005)
    if isinstance(llm_output, dict):
        return {
            **llm_output,
            "postprocessed": True,
            "timestamp": time.time()
        }
    return llm_output

production_pipeline = Chain(
    FunctionAgent(preprocessor_fn, name="Preprocessor"),
    full_stack,            # The resilience stack is just an agent — plug it in anywhere
    FunctionAgent(postprocessor_fn, name="Postprocessor"),
    name="ProductionChatbotPipeline"
)

print("Full production pipeline:")
print("  Preprocessor → [CircuitBreaker[Retry[Memory[LLM]]]] with HumanReview → Postprocessor")

async def demo_combined():
    print()
    print("=" * 60)
    print("COMBINED RESILIENCE STACK — 2-turn conversation")
    print("=" * 60)
    
    for i, msg in enumerate(["Hello, I'm testing the resilience stack.", "Tell me about circuit breakers."], 1):
        print(f"\nTurn {i}: {msg}")
        result = await production_pipeline.run(msg)
        print(f"  Output: {result}")

asyncio.run(demo_combined())

## Section 15: Real-World Example — Robust LLM Pipeline

Let's build a complete production-grade LLM pipeline with:
- **5 conversation turns** (memory-enabled)
- **Simulated API failures** at turns 2 and 3 (retry handles them)
- **Circuit breaker** monitoring the health
- **Human review** for outputs over a token threshold

In [ ]:
# --- Real-world robust LLM pipeline ---

# Simulated LLM API with known failure pattern
llm_call_log = []

async def flaky_llm_api(messages, context=None):
    """
    Simulates a real LLM API that:
    - Has occasional rate limit errors (2nd and 3rd calls)
    - Otherwise succeeds with context-aware responses
    """
    total_calls = len(llm_call_log)
    llm_call_log.append(messages)
    
    # Simulate rate limit on calls 2 and 3
    if total_calls in (1, 2):  # 0-indexed: calls 2 and 3
        raise ConnectionError(f"429 Rate limit exceeded (call #{total_calls + 1})")
    
    await asyncio.sleep(0.02)
    
    if isinstance(messages, dict):
        history = messages.get("history", [])
        msg = messages.get("message", "")
    else:
        history = []
        msg = str(messages)
    
    # Context-aware response (longer = more context)
    context_info = f" Based on {len(history)} previous turns: I know your context." if history else ""
    return {
        "response": f"Assistant: {msg[:50]}{context_info}",
        "model": "gpt-4-turbo",
        "tokens_used": 50 + len(history) * 30
    }

robust_base = FunctionAgent(flaky_llm_api, name="FlaklyLLMAPI")

# Build the full resilience stack for this demo
robust_memory  = MemoryAgent(agent=robust_base, window=5)
robust_retry   = RetryAgent(agent=robust_memory, max_attempts=4, base_delay=0.05, backoff_factor=2.0)
robust_cb      = CircuitBreakerAgent(agent=robust_retry, failure_threshold=5, recovery_timeout=5.0)

review_decisions = []

async def production_review_fn(output):
    """Production review: log and approve everything (monitoring mode)."""
    tokens = output.get("tokens_used", 0) if isinstance(output, dict) else 0
    decision = "APPROVED"
    review_decisions.append({"tokens": tokens, "decision": decision})
    if tokens > 200:
        print(f"    [Review] Flagged: {tokens} tokens — over threshold, but approved")
    return True

robust_hil = HumanInLoopAgent(agent=robust_cb, review_fn=production_review_fn, timeout=30.0)

async def run_robust_pipeline():
    print("=" * 70)
    print("ROBUST LLM PIPELINE — 5 turns with retry + circuit breaker + memory")
    print("=" * 70)
    print("Resilience stack: HumanInLoop(CircuitBreaker(Retry(Memory(LLM))))")
    print("Known failures: LLM API will return 429 on calls 2 and 3")
    print("RetryAgent will handle these transparently (max 4 attempts per turn)")
    print()
    
    conversation = [
        "Hello! I'm building a stock trading bot in Python.",
        "What data sources would you recommend for live prices?",
        "How should I handle API rate limits in my bot?",
        "What's the best architecture for low-latency order execution?",
        "Can you summarize our entire discussion?"
    ]
    
    results = []
    for turn_num, msg in enumerate(conversation, 1):
        print(f"Turn {turn_num}: {msg}")
        start = time.time()
        
        try:
            result = await robust_hil.run(msg)
            elapsed = time.time() - start
            results.append(result)
            
            response_preview = ""
            if isinstance(result, dict):
                response_preview = result.get("response", str(result))[:80]
                tokens = result.get("tokens_used", "?")
            else:
                response_preview = str(result)[:80]
                tokens = "?"
            
            print(f"  Response: {response_preview}")
            print(f"  Tokens:   {tokens}, Time: {elapsed*1000:.0f}ms, CB: {robust_cb.circuit_state}")
        
        except Exception as e:
            elapsed = time.time() - start
            print(f"  FAILED: {e} (in {elapsed*1000:.0f}ms)")
        
        print()
    
    print("=" * 70)
    print("PIPELINE SUMMARY")
    print("=" * 70)
    print(f"Turns completed:       {len(results)}/{len(conversation)}")
    print(f"Total LLM API calls:   {len(llm_call_log)} (includes retries)")
    print(f"Expected:              {len(conversation)} turns + 2 retried calls = {len(conversation)+2}")
    print(f"Circuit breaker state: {robust_cb.circuit_state}")
    print(f"Review decisions:      {len(review_decisions)} decisions made")
    print()
    
    print("Call log (each entry is one LLM API invocation):")
    for i, call in enumerate(llm_call_log, 1):
        if isinstance(call, dict):
            msg_preview = call.get("message", "")[:40]
            hist_len = len(call.get("history", []))
        else:
            msg_preview = str(call)[:40]
            hist_len = 0
        print(f"  API Call {i:2d}: msg='{msg_preview}' history={hist_len} turns")
    
    print()
    print("Key: Calls 2 and 3 triggered 429 errors.")
    print("     RetryAgent retried those turns automatically — user saw no failures.")

asyncio.run(run_robust_pipeline())

## Summary and Key Takeaways

In this notebook we built production-grade resilience into agentic AI systems using four primitives:

### Resilience Primitives Reference

| Primitive | Constructor | Key Params | When to Use |
|---|---|---|---|
| **RetryAgent** | `RetryAgent(agent, max_attempts=3, base_delay=1.0)` | `max_attempts`, `base_delay`, `backoff_factor` | Transient failures: rate limits, timeouts |
| **CircuitBreakerAgent** | `CircuitBreakerAgent(agent, failure_threshold=5)` | `failure_threshold`, `recovery_timeout` | Persistent failures: service outages |
| **MemoryAgent** | `MemoryAgent(agent, window=10)` | `window` | Multi-turn conversations |
| **HumanInLoopAgent** | `HumanInLoopAgent(agent, review_fn)` | `review_fn`, `timeout` | High-stakes decisions |

### Composition Rules

**Recommended ordering** (inner to outer):
```
HumanInLoop(
  CircuitBreaker(           ← Don't waste retries on a broken circuit
    Retry(                  ← Handle transient errors before counting as CB failure
      Memory(               ← Add context closest to the LLM
        base_llm_agent
      )
    )
  )
)
```

### Production Checklist

For any LLM agent going to production:
- [ ] Wrapped in `RetryAgent` with `max_attempts >= 3`
- [ ] Wrapped in `CircuitBreakerAgent` with reasonable `failure_threshold`
- [ ] If multi-turn: wrapped in `MemoryAgent` with appropriate `window`
- [ ] If consequential: wrapped in `HumanInLoopAgent` with a proper `review_fn`
- [ ] Run through `Runtime` for observability and stats

### Next Steps

You've now completed the **Multigen SDK Tutorial Series**!

Previous notebooks:
- **Notebook 16**: MCMC Probabilistic State Machines
- **Notebook 17**: Event Bus and Messaging
- **Notebook 18**: Runtime — Unified Execution with Observability
- **Notebook 19**: This notebook — Resilience Patterns

For production deployment, combine all four: state machines for complex workflows, event bus for agent communication, runtime for observability, and resilience wrappers for reliability.